### Llama Guard 8B moderation


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "notebooks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'meta-llama/Meta-Llama-Guard-2-8B'
MAX_LENGTH = 512
device = "cuda" if torch.cuda.is_available() else "cpu"
sns.set_theme(style="whitegrid")


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


def load_text_frame() -> pd.DataFrame:
    for candidate in [Path(path) for path in DATASET_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.suffix == ".parquet":
            frame = pd.read_parquet(candidate)
        else:
            frame = pd.read_csv(candidate)
        break
    else:
        raise FileNotFoundError("No text moderation dataset was found.")

    if "text" not in frame.columns:
        text_like = [col for col in frame.columns if "text" in col.lower() or "comment" in col.lower()]
        frame["text"] = frame[text_like[0]]
    if "label" not in frame.columns:
        label_like = [col for col in frame.columns if "label" in col.lower() or "toxic" in col.lower()]
        frame["label"] = frame[label_like[0]] if label_like else 0
    frame["label"] = (pd.to_numeric(frame["label"], errors="coerce").fillna(0) >= 0.5).astype(int)
    frame["text"] = frame["text"].astype(str)
    return frame[["text", "label"]].drop_duplicates().reset_index(drop=True)


def build_prompt(text: str) -> str:
    return (
        "<s>[INST] You are a safety classifier. "
        "Label the following user message as SAFE or UNSAFE and briefly cite the violated policy categories if unsafe.\n\n"
        f"User message: {text} [/INST]"
    )


text_df = load_text_frame()
train_df, valid_df = train_test_split(
    text_df,
    test_size=0.2 if len(text_df) >= 50 else 0.3,
    stratify=text_df["label"] if text_df["label"].nunique() > 1 else None,
    random_state=42,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device).eval()


@torch.inference_mode()
def generate_labels(texts: list[str], max_new_tokens: int = 48):
    prompts = [build_prompt(text) for text in texts]
    batch = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    generated = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.batch_decode(generated[:, batch["input_ids"].shape[1] :], skip_special_tokens=True)
    return decoded


preview = valid_df.head(min(8, len(valid_df))).copy()
preview["response"] = generate_labels(preview["text"].tolist())
preview["score"] = preview["response"].str.contains("unsafe", case=False, regex=False).astype(float)
display(preview)


In [ ]:
inspection_batch = tokenizer([build_prompt(text) for text in valid_df["text"].head(min(4, len(valid_df)))], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
with torch.no_grad():
    inspection_outputs = model(**inspection_batch, output_hidden_states=True)

hidden_norms = torch.stack([layer.norm(dim=-1).mean(dim=1).cpu() for layer in inspection_outputs.hidden_states], dim=0).numpy()
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(hidden_norms, cmap="mako", ax=ax)
plt.tight_layout()


In [ ]:
def build_instruction_dataset(frame: pd.DataFrame) -> Dataset:
    data = []
    for _, row in frame.iterrows():
        answer = "UNSAFE" if row["label"] == 1 else "SAFE"
        policy = "toxicity" if row["label"] == 1 else "none"
        text = build_prompt(row["text"]) + f" {answer} | policy={policy}</s>"
        data.append({"text": text})
    return Dataset.from_list(data)


train_dataset = build_instruction_dataset(train_df)
valid_dataset = build_instruction_dataset(valid_df)


def tokenize_for_sft(batch):
    tokens = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


train_tokens = train_dataset.map(tokenize_for_sft, batched=True, remove_columns=["text"])
valid_tokens = valid_dataset.map(tokenize_for_sft, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir=str(ARTIFACT_ROOT / "llama_guard_base"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_tokens, eval_dataset=valid_tokens, tokenizer=tokenizer)
# trainer.train()  # Enable when GPU memory is available.
tokenizer.save_pretrained(ARTIFACT_ROOT / "llama_guard_base")


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


def parse_unsafe_score(response: str) -> float:
    response = response.lower()
    if "unsafe" in response:
        return 0.95
    if "safe" in response:
        return 0.05
    return 0.5


baseline_responses = generate_labels(valid_df["text"].head(min(32, len(valid_df))).tolist())
baseline_scores = np.array([parse_unsafe_score(resp) for resp in baseline_responses])
base_metrics = compute_binary_metrics(valid_df["label"].head(len(baseline_scores)), baseline_scores, threshold=0.5)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_dora=False,
)
dora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_dora=True,
)

lora_model = get_peft_model(AutoModelForCausalLM.from_pretrained(MODEL_ID), lora_cfg)
dora_model = get_peft_model(AutoModelForCausalLM.from_pretrained(MODEL_ID), dora_cfg)
lora_model.save_pretrained(ARTIFACT_ROOT / "llama_guard_lora")
dora_model.save_pretrained(ARTIFACT_ROOT / "llama_guard_dora")
pd.DataFrame([{"run": "prompt_baseline", **base_metrics}])
